In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

df = pd.read_csv('data/dataset.csv')
print(f'Dataset: {df.shape}')

In [ ]:
print(f'Valeurs manquantes: {df.isnull().sum().sum()}')
print(f'Doublons: {df.duplicated().sum()}')
print(f'\nDistribution cible:\n{df["category"].value_counts()}')

In [ ]:
cols = ['age', 'region', 'subscription', 'months', 'mobility', 'category']
df_enc = pd.get_dummies(df[cols])

X = df_enc.drop(columns=[c for c in df_enc.columns if c.startswith('category_')])
y = df['category']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('Standardisation appliquée')

In [ ]:
model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train_scaled, y_train)
print('Modèle entraîné')

In [ ]:
y_pred = model.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc:.2%}')

cm = confusion_matrix(y_test, y_pred, labels=model.classes_)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=model.classes_,
            yticklabels=model.classes_)
plt.title('Matrice de confusion')
plt.xlabel('Prédit')
plt.ylabel('Réel')
plt.tight_layout()
plt.savefig('outputs/confusion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
profiles = pd.DataFrame({
    'age': ['80+', '60-69', '70-79'],
    'region': ['Île-de-France', 'PACA', 'Bretagne'],
    'subscription': ['Basique', 'Premium', 'Basique'],
    'months': [3, 48, 20],
    'mobility': ['Faible', 'Élevée', 'Moyenne']
})

profiles_enc = pd.get_dummies(profiles).reindex(columns=X.columns, fill_value=0)
profiles_scaled = scaler.transform(profiles_enc)
preds = model.predict(profiles_scaled)

print('Prédictions test:')
for i, p in enumerate(preds):
    print(f'Profil {i+1}: {p}')